# 07 — The evaluation layer (stage 8)

**Purpose.** One evaluation code for every model of the project: metrics, confidence intervals,
Diebold–Mariano tests, seasons, operating conditions, the comparison of the two routes, and the
reference forecasts that every model must beat.

**Two parts.**
- **Part A** proves that the code computes what the original computed (T-15), on the only
  predictions of the original that survived.
- **Part B** moves the truth to the corrected feature tables and adds the references.

**Rules.**
- The truth comes from the stage-6 tables, never from the prediction file.
- Every model is scored on every daytime hour that has a truth (C-35).
- The two routes are compared on the same hours (C-34).
- **The test split is locked** until the final evaluation after gate G-1.

**Documented in** OP-30, OP-40, OP-41 and OP-54 (the kW statistics), `docs/stage8b_plan.md`,
run journal R8-01 to R8-03 and R9-11, tests T-03, T-04, T-12, T-15, T-38 to T-41 and T-65.

## 1. Part A: the code reproduces the original numbers (T-15)

In [1]:
import pandas as pd

from solarpv.config.paths import EVIDENCE_DIR, OUTPUTS_DIR

old = pd.read_csv(EVIDENCE_DIR / "01_benchmark" / "benchmark_summary.csv")
new = pd.read_csv(OUTPUTS_DIR / "07_evaluation_fixture" / "benchmark_summary.csv")
t15 = new.merge(old[old["model"] == "lstm_sota"], on=["model", "target", "horizon"],
                suffixes=("_rebuild", "_submitted"))
t15[["target", "horizon", "n_rebuild", "n_submitted", "nRMSE_pct_rebuild",
     "nRMSE_pct_submitted"]]

,target,horizon,n_rebuild,n_submitted,nRMSE_pct_rebuild,nRMSE_pct_submitted
0,direct,1,2510,2510,12.576248,12.5774
1,direct,2,2512,2512,12.636702,12.6371
2,direct,3,2514,2514,12.647214,12.6478
3,indirect,1,2510,2510,30.237345,30.2374
4,indirect,2,2512,2512,30.460159,30.4602
5,indirect,3,2514,2514,30.530334,30.5303


**What it shows.** The surviving BiLSTM predictions of the original, evaluated by the new
code with the original's conventions (its own truth, its daytime mask and its MEAN_K), next to the
submitted benchmark.

**What it means.** The numbers agree to at most 0.0012 points of nRMSE, on the
same number of hours. The evaluation code is right; every later difference from the report comes
from the data and the corrections, not from the metric code.

## 2. The test split is locked

In [2]:
from solarpv.evaluation import evaluate

try:
    evaluate.run([], OUTPUTS_DIR / "scratch", mean_k=0.788157, truth=pd.DataFrame(),
                 split="test")
except PermissionError as error:
    print("refused:", error)

refused: the test split is evaluated only with allow_test=True (final evaluation, after g1-frozen)


**What it shows.** Asking the evaluation for the test split is refused unless the caller
passes `allow_test=True`. That flag is used once, in the final evaluation after the code is
frozen (`g1-frozen`), and every use is written to `outputs/07_evaluation/test_access_log.csv`.

**What it means.** No decision of the project, from the thresholds of stage 5 to the features of
stage 7 and the tuning of stage 9, can see the test results. In the original, test results were
read throughout development.

## 3. Part B: the references on validation

In [3]:
bench = pd.read_csv(OUTPUTS_DIR / "07_evaluation" / "val" / "benchmark_summary.csv")
bench.pivot_table(index=["model", "target"], columns="horizon",
                  values="nRMSE_kW_pct").round(2)

horizon                  1      2      3
model       target                      
clearsky_k  direct    8.96   8.96   8.96
            indirect  8.96   8.96   8.96
climatology direct    8.04   8.04   8.04
            indirect  8.06   8.06   8.06
persistence direct    9.68  10.82  10.72
            indirect  9.68  10.84  10.76

**What it shows.** nRMSE in % of the 3,449 kW capacity on the 2,901
daytime validation hours, for the three reference forecasts:
- **Persistence:** the same hour of the last day known at T0.
- **Climatology:** the train mean of each month and hour.
- **Clear sky:** the clear-sky power times the mean daytime k.

**What it means.** At D+1 for the direct target:
- climatology 8.04 %;
- clear sky 8.96 %;
- persistence 9.68 %.

In this desert climate persistence is the weakest reference, and climatology the strongest. A
model that does not beat climatology adds nothing; that is the bar for stage 9. The errors of
both routes are given in kW, so the routes compare directly.

In [4]:
views = pd.read_csv(OUTPUTS_DIR / "07_evaluation" / "val" / "benchmark_views.csv")
views[(views["horizon"] == 1)].pivot_table(index=["model", "target"], columns="view",
                                           values="nRMSE_kW_pct").round(2)

view                   all  no_polled  normal_operation  source_minute
model       target                                                    
clearsky_k  direct    8.96       8.96              7.68           8.96
            indirect  8.96       8.96              7.68           8.96
climatology direct    8.04       8.04              6.76           8.04
            indirect  8.06       8.06              6.77           8.06
persistence direct    9.68       9.68              8.59           9.68
            indirect  9.68       9.68              8.59           9.68

**What it shows.** The same scores on subsets of hours (D-21, D-22): all hours, normal
operation only (no stop, curtailment or derate), and by data source.

**What it means.** For climatology at D+1 (direct), excluding the op-limited hours lowers the
error from 8.04 % to 6.76 %. Part of every error is the plant,
not the weather; the main result uses all hours, and this view shows the share. The views of the
2025 source (polled, mixed) do not appear on validation: that source exists only in the test
months.

In [5]:
dm = pd.read_csv(OUTPUTS_DIR / "07_evaluation" / "val" / "dm_tests.csv")
dm[["target", "horizon", "model_a", "model_b", "dm_stat_hln", "p_value_hln",
    "pct_of_mde", "significant"]].round(3)

,target,horizon,model_a,model_b,dm_stat_hln,p_value_hln,pct_of_mde,significant
0,direct,1,clearsky_k,persistence,-1.425,0.154,50.233,False
1,direct,1,climatology,persistence,-3.231,0.001,109.470,True
2,direct,2,clearsky_k,persistence,-2.980,0.003,96.773,True
3,direct,2,climatology,persistence,-4.359,0.000,136.282,True
4,direct,3,clearsky_k,persistence,-2.706,0.007,84.663,True
5,direct,3,climatology,persistence,-4.033,0.000,116.952,True
6,indirect,1,clearsky_k,persistence,-0.785,0.433,22.315,False
7,indirect,1,climatology,persistence,-3.310,0.001,95.192,True
8,indirect,2,clearsky_k,persistence,-2.480,0.013,70.277,True
9,indirect,2,climatology,persistence,-4.305,0.000,127.797,True


**What it shows.** Diebold–Mariano tests against persistence, with the Harvey–Leybourne–
Newbold correction and Bonferroni, and the share of the minimum detectable effect.

**What it means.** 10 of 12 comparisons are significant at the
corrected level. The original report described HLN and Bonferroni but computed neither (C-15);
here both are computed, and the uncorrected value is reported next to them.

In [6]:
dm_kw = pd.read_csv(OUTPUTS_DIR / "07_evaluation" / "val" / "dm_tests_kw.csv")
dm_kw[["target", "horizon", "model_a", "model_b", "dm_stat_hln", "p_value_hln",
       "pct_of_mde", "significant"]].round(3)

,target,horizon,model_a,model_b,dm_stat_hln,p_value_hln,pct_of_mde,significant
0,direct,1,clearsky_k,persistence,-1.425,0.154,50.233,False
1,direct,1,climatology,persistence,-3.231,0.001,109.470,True
2,direct,2,clearsky_k,persistence,-2.980,0.003,96.773,True
3,direct,2,climatology,persistence,-4.359,0.000,136.282,True
4,direct,3,clearsky_k,persistence,-2.706,0.007,84.663,True
5,direct,3,climatology,persistence,-4.033,0.000,116.952,True
6,indirect,1,clearsky_k,persistence,-1.442,0.149,50.754,False
7,indirect,1,climatology,persistence,-3.193,0.001,108.947,True
8,indirect,2,clearsky_k,persistence,-3.024,0.003,97.375,True
9,indirect,2,climatology,persistence,-4.354,0.000,135.315,True


**What it shows.** The same tests on the errors in kW against the actual AC power, for
both routes (Q-19, added on 2026-11-07). The indirect forecast is converted with the same
function as every other kW number.

**What it means.** 10 of 12 comparisons are significant. For
the direct target the two tables agree (largest difference of the statistic
0), because its unit is already kW. For the indirect route the k table gives every
daytime hour the same weight, so the low-sun hours, where k is unstable and the power small, weigh
as much as noon; the kW table weighs the hours by their power. Every decision of the project is
taken in kW, so the kW table is the one stage 10 reads, and the k table stays as a secondary
statistic.

In [7]:
seasons = pd.read_csv(OUTPUTS_DIR / "07_evaluation" / "val" / "seasonal_metrics.csv")
d1 = seasons[(seasons["horizon"] == 1) & (seasons["target"] == "direct")]
d1.pivot_table(index="model", columns="season", values="nRMSE_pct").round(2)

season,Fall,Spring,Summer,Winter
model,,,,
clearsky_k,5.12,10.94,5.13,14.48
climatology,4.10,10.19,2.87,14.07
persistence,4.83,12.73,2.52,16.27


**What it shows.** D+1 errors of the direct target by season on validation (February to
September 2024), nRMSE in % of capacity.

**What it means.** Errors are largest in the seasons with changeable weather and smallest in the
clear summer. The same tables will be produced for every model in stages 9 to 11.

**Run recorded in** `outputs/07_evaluation/runs/20261107T135905Z.log` (commit f529913, truth
`data/processed/06_features/features_1H_D1.parquet`).

**Next stages.** Stage 7 selects the features; stage 9 trains the light models and scores them
here against these references.

**Reproduce.** `python -m solarpv.evaluation.evaluate --fixture`,
`python -m solarpv.evaluation.evaluate`, then `python scripts/make_evaluation_notebook.py`.